# Assignment 2: Retrieval-Augmented Generation Question Answering

Welcome to the second assignment for 50.055 Machine Learning Operations This is a continuation of your work in assignment 1, where you will be working to build a chatbot which can answer questions about SUTD to prospective students.


**This assignment is a individual assignment.**

- Read the instructions in this notebook carefully
- Add your solution code and answers in the appropriate places. The questions are marked as **QUESTION:**, the places where you need to add your code and text answers are marked as **ADD YOUR SOLUTION HERE**
- The completed notebook, including your added code and generated output will be your submission for the assignment.
- The notebook should execute without errors from start to finish when you select "Restart Kernel and Run All Cells..". Please test this before submission.
- Use the SUTD Education Cluster to solve and test the assignment. If you work on another environment, minimally test your work on the SUTD Education Cluster.

**Rubric for assessment** 

Your submission will be graded using the following criteria. 
1. Code executes: your code should execute without errors. The SUTD Education cluster should be used to ensure the same execution environment.
2. Correctness: the code should produce the correct result or the text answer should state the factual correct answer.
3. Style: your code should be written in a way that is clean and efficient. Your text answers should be relevant, concise and easy to understand.
4. Partial marks will be awarded for partially correct solutions.
5. Creativity and innovation: in this assignment you have more freedom to design your solution, compared to the first assignments. You can show of your creativity and innovative mindset. 
6. There is a maximum of 215 points for this assignment, which will be normalized to 10% of your grade.

**ChatGPT policy** 

If you use AI tools, such as ChatGPT, Claude, Gemini, etc. to solve the assignment questions, you need to be transparent about its use and mark AI-generated content as such. In particular, you should include the following in addition to your final answer:
- A copy or screenshot of the prompt you used
- The name of the AI model
- The AI generated output
- An explanation why the answer is correct or what you had to change to arrive at the correct answer

**Assignment Notes:** Please make sure to save the notebook as you go along. Submission instructions are located at the bottom of the notebook.



### Retrieval-Augmented Generation (RAG) 

In this assignment, you will be building a Retrieval-Augmented Generation (RAG) question answering system which can answer questions about SUTD.

We'll be leveraging `langchain` and the LLM API of your choice from assignment 1

Check out the docs:
- [LangChain](https://docs.langchain.com/docs/)


The SUTD website used to allow chatting with current students. Unfortunately, this feature does not exist anymore. Let's build a chatbot to fill this gap!


# Install dependencies
Use pip to install all required dependencies of this assignment in the cell below. Make sure to test this on the SUTD cluster as different environments have different software pre-installed.  

Use of AI is documented and attached in a link here as I do not want to mess up the notebook:

link.com

In [1]:
# QUESTION: Install and import all required packages
# The rest of your code should execute without any import or dependency errors.

# **--- ADD YOUR SOLUTION HERE (10 points) ---**

%pip install -q requests beautifulsoup4 pandas

# this was ran aft chunking
%pip install -q langchain langchain-community sentence-transformers faiss-cpu transformers torch accelerate

# ran before llm
%pip install -q openai python-dotenv

# ran before putting it all tgt
%pip install -q langchain-openai

# ----------------


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import requests, bs4, pandas


# Download documents
The RAG application should be able to answer questions based on ingested documents. For the SUTD chatbot, download PDF and HTML files from the SUTD website. The documents should contain information about the admission process, available courses and the university in general.


As my initial

In [92]:
# QUESTION: Download documents from the SUTD website
# You should download at least 10 documents but more documents can increase the knowledge base of your chatbot.

# **--- ADD YOUR SOLUTION HERE (20 points) ---**

RUN_SCRAPING = False
RUN_EXTRACTION = False

"""Explanation for this is as follows:
I added these toggles to control whether the scraping and extraction .py scripts run. These scripts were the exact same ones used in Assignment 1
If these steps are executed every time the notebook runs, the same webpages will be scraped again and again
leading to duplicate entries
For now, it is set to false as all files have been downloaded and processed.
"""


In [89]:
from pathlib import Path

if RUN_SCRAPING:
    !python -m src.fetch_html \
      --seed data/seed_urls_pages.txt \
      --out data/raw/pages \
      --meta data/raw/pages/metadata.csv \
      --delay 2
else:
    print("Skipping HTML fetching (RUN_SCRAPING=False)")

raw_files = list(Path("data/raw/pages").glob("*.html"))

print(f"Total HTML files fetched: {len(raw_files)}\n")

Skipping HTML fetching (RUN_SCRAPING=False)
Total HTML files fetched: 22



In [90]:
if RUN_EXTRACTION:
    !python -m src.extract_page_content \
      --raw data/raw/pages \
      --out data/processed/docs \
      --meta data/processed/docs/metadata.csv
else:
    print("Skipping content extraction (RUN_EXTRACTION=False)")

docs = list(Path("data/processed/docs").rglob("*.txt"))

print(f"Total cleaned documents created: {len(docs)}\n")

Skipping content extraction (RUN_EXTRACTION=False)
Total cleaned documents created: 23



# Split documents
Use LangChain to split the documents into smaller text chunks. 

In [3]:
# building registry

from pathlib import Path
import pandas as pd

raw_meta = pd.read_csv("data/raw/pages/metadata.csv")

raw_meta["raw_file"] = raw_meta["raw_path"].apply(lambda x: Path(x).name if pd.notna(x) else "")
raw_lookup = raw_meta.set_index("raw_file")[["url", "retrieved_at"]].to_dict("index")

rows = []

base = Path("data/processed/docs")

for domain_dir in ["academics", "admissions", "housing"]:
    folder = base / domain_dir

    for fp in sorted(folder.glob("*_content.txt")):
        raw_file = fp.name.replace("_content.txt", ".html")
        src = raw_lookup.get(raw_file, {})
        text = fp.read_text(encoding="utf-8")
        rows.append({
            "doc_id": fp.stem,
            "domain": domain_dir,
            "path": str(fp),
            "source_url": src.get("url", ""),
            "retrieved_at": src.get("retrieved_at", ""),
            "text": text
        })

doc_registry = pd.DataFrame(rows)

print("Total documents:", len(doc_registry))
doc_registry.head()

Total documents: 22


,doc_id,domain,path,source_url,retrieved_at,text
0,about_design-ai_content,academics,data/processed/docs/academics/about_design-ai_...,https://www.sutd.edu.sg/about/design-ai/,2026-03-14T08:56:49.833205+00:00,About\nDesign·AI\nAbout\nDesign·AI – Innovatio...
1,asd_education_undergraduate_curriculum_content,academics,data/processed/docs/academics/asd_education_un...,https://www.sutd.edu.sg/asd/education/undergra...,2026-03-14T09:15:06.623893+00:00,Education\nUndergraduate programme\nASD curric...
2,dai_education_undergraduate_curriculum_content,academics,data/processed/docs/academics/dai_education_un...,https://www.sutd.edu.sg/dai/education/undergra...,2026-03-14T09:15:10.901466+00:00,Education\nUndergraduate programme\nDAI curric...
3,education_undergraduate_content,academics,data/processed/docs/academics/education_underg...,https://www.sutd.edu.sg/education/undergraduate,2026-02-24T16:53:09.057783+00:00,Design·AI Education\nUndergraduate studies at ...
4,education_undergraduate_freshmore-subjects_con...,academics,data/processed/docs/academics/education_underg...,https://www.sutd.edu.sg/education/undergraduat...,2026-02-24T16:53:11.186665+00:00,Design·AI Education\nUndergraduate studies at ...


In [4]:
docs = []

for _, row in doc_registry.iterrows():
    text = Path(row["path"]).read_text(encoding="utf-8")
    docs.append({
        "doc_id": row["doc_id"],
        "domain": row["domain"],
        "source_url": row["source_url"],
        "retrieved_at": row["retrieved_at"],
        "text": text
    })

print(f"Loaded {len(docs)} documents.")
for i, doc in enumerate(docs, 1):
    print(f"{i}. {doc['doc_id']} - {doc['domain']}")

Loaded 22 documents.
1. about_design-ai_content - academics
2. asd_education_undergraduate_curriculum_content - academics
3. dai_education_undergraduate_curriculum_content - academics
4. education_undergraduate_content - academics
5. education_undergraduate_freshmore-subjects_content - academics
6. epd_education_undergraduate_curriculum_content - academics
7. esd_education_undergraduate_curriculum_beyond-ay-2019_content - academics
8. istd_education_undergraduate_curriculum_content - academics
9. media-releases-listing_sutd-becomes-worlds-first-university-to-incorporate-design-ai-in-all-first-year-courses_content - academics
10. media-releases-listing_sutd-broadens-scope-of-flagship-design-and-ai-degree-first-university-to-integrate-social-sciences-into-technology-degree_content - academics
11. admissions_undergraduate_admission-requirements_international-qualifications_content - admissions
12. admissions_undergraduate_admission-requirements_overview_content - admissions
13. admissions

In [5]:
# doc length statistics (shows smallest, biggest and avg)
doc_registry["length"] = doc_registry["text"].apply(len)

print(doc_registry["length"].describe())

count       22.000000
mean      6139.590909
std       4342.285581
min       1172.000000
25%       2446.500000
50%       4991.500000
75%       8947.500000
max      15794.000000
Name: length, dtype: float64


In [6]:
doc_registry[["doc_id", "length"]].sort_values("length", ascending=False).head(5)


,doc_id,length
6,esd_education_undergraduate_curriculum_beyond-...,15794
2,dai_education_undergraduate_curriculum_content,12541
7,istd_education_undergraduate_curriculum_content,12439
1,asd_education_undergraduate_curriculum_content,11468
14,admissions_undergraduate_education-expenses_fe...,10282


In [33]:

# QUESTION: Use langchain to split the documents into chunks 

#--- ADD YOUR SOLUTION HERE (20 points)---


from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
import pandas as pd

documents = [
    Document(
        page_content=row["text"],
        metadata={
            "doc_id": row["doc_id"],
            "domain": row["domain"],
            "source_url": row["source_url"],
            "retrieved_at": row["retrieved_at"],
            "path": row["path"],
        }
    )
    for _, row in doc_registry.iterrows()
]

# langchain text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200, ### initial: 1000
    chunk_overlap=150, ### initial: 150
    separators=["\n\n", "\n", " ", ""]
)
chunked_docs = text_splitter.split_documents(documents)

# make into table
chunk_rows = []
for i, doc in enumerate(chunked_docs):
    chunk_rows.append({
        "chunk_id": f"{doc.metadata['doc_id']}_chunk_{i:03d}",
        "doc_id": doc.metadata["doc_id"],
        "domain": doc.metadata["domain"],
        "source_url": doc.metadata["source_url"],
        "retrieved_at": doc.metadata["retrieved_at"],
        "path": doc.metadata["path"],
        "text": doc.page_content,
        "chunk_length": len(doc.page_content),
    })

chunk_df = pd.DataFrame(chunk_rows)

print("Total documents:", len(documents))
print("Total chunks:", len(chunked_docs))
chunk_df.head(3)

Total documents: 22
Total chunks: 140


,chunk_id,doc_id,domain,source_url,retrieved_at,path,text,chunk_length
0,about_design-ai_content_chunk_000,about_design-ai_content,academics,https://www.sutd.edu.sg/about/design-ai/,2026-03-14T08:56:49.833205+00:00,data/processed/docs/academics/about_design-ai_...,About\nDesign·AI\nAbout\nDesign·AI – Innovatio...,967
1,about_design-ai_content_chunk_001,about_design-ai_content,academics,https://www.sutd.edu.sg/about/design-ai/,2026-03-14T08:56:49.833205+00:00,data/processed/docs/academics/about_design-ai_...,By going beyond its traditional role as a tech...,846
2,about_design-ai_content_chunk_002,about_design-ai_content,academics,https://www.sutd.edu.sg/about/design-ai/,2026-03-14T08:56:49.833205+00:00,data/processed/docs/academics/about_design-ai_...,SUTD’s Design·AI principle puts the human back...,768


In [34]:
# QUESTION: create embeddings of document chunks and store them in a local vector store for fast lookup
# Decide an appropriate embedding model. Use Huggingface to run the embedding model locally.
# You do not have to use cloud-based APIs.

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

langchain_docs = chunked_docs

print(f"Total LangChain documents: {len(langchain_docs)}")

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = FAISS.from_documents(langchain_docs, embedding_model)

print("FAISS vector store created successfully.")
print(f"Total chunks stored: {len(langchain_docs)}")

vectorstore.save_local("data/vectorstore/faiss_index")
print("Vector store saved to data/vectorstore/faiss_index")


Total LangChain documents: 140


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6943.54it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FAISS vector store created successfully.
Total chunks stored: 140
Vector store saved to data/vectorstore/faiss_index


I used the HuggingFace embedding model sentence-transformers/all-MiniLM-L6-v2 together with the FAISS vector store. I chose this embedding model because it is lightweight, fast, runs locally, and is widely used for semantic retrieval tasks. This makes it suitable for an assignment setting where reproducibility and efficiency matter. I used FAISS because it is a simple and efficient local vector store for nearest-neighbour similarity search. It integrates well with LangChain and does not require any external service. Since the document collection is relatively small, FAISS is more than sufficient for fast lookup. This setup keeps the RAG pipeline fully local and easy to run on a laptop or cluster.

### QUESTION: 

What chunking method or strategy did you use? Why did you use this method. Explain your design decision in less than 10 sentences.


**--- ADD YOUR SOLUTION HERE (10 points) ---**


------------------------------


In [ ]:
# QUESTION: create embeddings of document chunks and store them in a local vector store for fast lookup
# Decide an appropriate embedding model. Use Huggingface to run the embedding model locally.
# You do not have to use cloud-based APIs.

#--- ADD YOUR SOLUTION HERE (20 points)---


#------------------------------


### QUESTION: 

What embeddings and vector store did you use and why? Explain your design decision in less than 10 sentences.


**--- ADD YOUR SOLUTION HERE (10 points) ---**


------------------------------



In [22]:
# Execute a query against the vector store

query = "When was SUTD founded?"

# QUESTION: run the query against the vector store, print the top 5 search results

results = vectorstore.similarity_search(query, k=5)

print(f"Top {len(results)} results for query: {query}\n")

for i, doc in enumerate(results, start=1):
    print("=" * 120)
    print(f"Result {i}")
    print("Metadata:", doc.metadata)
    print("Content preview:")
    print(doc.page_content[:800])
    print()

Top 5 results for query: When was SUTD founded?

Result 1
Metadata: {'doc_id': 'admissions_undergraduate_local-diploma_application-timeline_content', 'domain': 'admissions', 'source_url': 'https://www.sutd.edu.sg/admissions/undergraduate/local-diploma/application-timeline/#tabs', 'retrieved_at': '2026-03-14T08:56:56.209306+00:00', 'path': 'data/processed/docs/admissions/admissions_undergraduate_local-diploma_application-timeline_content.txt'}
Content preview:
Are all diplomas accepted for admission?
Generally, diplomas from the School of Engineering, Information Technology, Architecture or Sciences are considered more relevant to SUTD’s courses and hence will be assessed more favourably for admission. However, other diplomas may still be considered on a case-by-case basis.
Can’t find what you need?
View all FAQs
Ask admissions
Submit an enquiry or schedule a call with our friendly admissions team.
Reach out to us
Take a virtual tour
Step through your screen and into SUTD, with immersiv

In [23]:
# Execute the below query against the model and let it it answer from it's internal memory

query = "What courses are available in SUTD?"


#--- ADD YOUR SOLUTION HERE (40 points)---
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(Path(".env"))
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not found in .env"

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# use same parameters as assignment 1
def openai_text(prompt: str, model: str = "gpt-4o-mini", max_output_tokens: int = 300, temperature: float = 0.2) -> str:
    resp = client.responses.create(
        model=model,
        input=prompt,
        max_output_tokens=max_output_tokens,
        temperature=temperature,
    )
    return resp.output_text

base_prompt = f"""You are a helpful assistant answering questions about SUTD.

Answer the question as accurately as possible using only your own internal knowledge.
If you are unsure, say you are unsure, do not fabricate something.

Question: {query}

Answer:"""

base_answer = openai_text(base_prompt, model="gpt-4o-mini", max_output_tokens=200, temperature=0.2)

print("Question:", query)
print("\nBase model answer (without RAG):\n")
print(base_answer)

#------------------------------


Question: What courses are available in SUTD?

Base model answer (without RAG):

The Singapore University of Technology and Design (SUTD) offers a range of courses primarily focused on design, technology, and engineering. The main pillars of their curriculum include:

1. **Architecture and Sustainable Design (ASD)**
2. **Engineering Product Development (EPD)**
3. **Information Systems Technology and Design (ISTD)**
4. **Engineering Systems and Design (ESD)**

Each of these pillars includes various core and elective courses that cover topics such as design thinking, product development, software engineering, systems engineering, and more. Additionally, SUTD emphasizes interdisciplinary learning and often integrates courses from different pillars.

For the most accurate and up-to-date information, it's best to refer to SUTD's official website or course catalog.


In [14]:
# QUESTION: Now put everything together. Use langchain to integrate your vector store and Llama model into a RAG system
# Run the below example question against your RAG system.

# Example questions
query = "How can I increase my chances of admission into SUTD?"


#--- ADD YOUR SOLUTION HERE (40 points)---


#------------------------------


In [24]:
# langchain llm wrapper
from langchain_core.language_models import LLM

class OpenAIWrapper(LLM):
    
    @property
    def _llm_type(self):
        return "custom_openai"

    def _call(self, prompt, stop=None):
        return openai_text(prompt)

llm = OpenAIWrapper()

In [25]:
# retriever from faiss

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 5}
)

In [36]:
from langchain_core.prompts import PromptTemplate

rag_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are a helpful assistant answering questions about SUTD.

Use the retrieved context below to answer the question as accurately as possible.
If the answer is partially available, provide the best supported answer based on the context.
If the answer is truly not present, say that the information is not available in the provided documents.
"""
)

In [27]:
# Example question
query = "How can I increase my chances of admission into SUTD?"

retrieved_docs = retriever.invoke(query)
context = "\n\n".join([doc.page_content for doc in retrieved_docs])

# Build RAG
rag_prompt_text = rag_prompt.format(
    context=context,
    question=query
)

rag_answer = openai_text(rag_prompt_text)

print("Question:", query)
print("\nRAG Answer:\n")
print(rag_answer)

Question: How can I increase my chances of admission into SUTD?

RAG Answer:

To increase your chances of admission into SUTD, consider the following strategies:

1. **Holistic Application**: Focus on presenting a well-rounded application that highlights not just your academic achievements but also your qualities and traits. SUTD values a growth mindset, collaboration, and curiosity/open-mindedness.

2. **Academic Competency**: Ensure strong performance in Mathematics and the Sciences, as these are key areas of focus. Your final year exam results and academic performance over the 2-3 years prior will be assessed.

3. **Relevant Qualifications**: If you have a diploma, aim for one from the School of Engineering, Information Technology, Architecture, or Sciences, as these are viewed more favorably. However, other diplomas may still be considered.

4. **Co-Curricular Activities**: Include a portfolio that showcases your involvement in co-curricular activities, as these can demonstrate you

In [43]:

# QUESTION: Below is set of test questions. Add another 10 test questions based on your user interviews and your value proposition canvas.
# Run the complete set of test questions against the RAG question answering system. 

questions = ["What are the admissions deadlines for SUTD?",
             "Is there financial aid available?",
             "What is the minimum score for the Mother Tongue Language?",
             "Do I require reference letters?",
             "Can polytechnic diploma students apply?",
             "Do I need SAT score?",
             "How many PhD students does SUTD have?",
             "How much are the tuition fees for Singaporeans?",
             "How much are the tuition fees for international students?",
             "Is there a minimum CAP?",
# added 10 here
            "What do I actually need to submit for my portfolio?",
            "Is it possible to get into SUTD with a polytechnic diploma?",
            "When will I find out if I got an interview?",
            "How are roommates matched in the hostel?",
            "What does the school like out for in applicants?",
            "How much does it cost to study there per year?",
            "If I don't have a good MTL grade, can I still get in?",
            "Does SUTD admissions rely solely on academic results?",
            "What exactly is the Computer Science course in SUTD about?",
            "What if my English is not good?"

            # added in qns from ass1 for testing. commented out aft refinement
            # "What subjects are common to all Freshmore students?",
            # "How are grades structured in Term 1 of the Freshmore curriculum?",
            # "What is the purpose of the computational thinking courses?",
            # "What is emphasized in the Introduction to Human-Centred Design course?",
            # "What are the two versions of the Math course offered?",
            # "What is the significance of the interdisciplinary design projects?",
            # "What subjects are included in Term 1 of the Freshmore curriculum?",
            # "What is the grading structure for subjects in Term 1?",
            # "What computational thinking courses are offered in the Freshmore curriculum?",
            # "What is the focus of the Design Computation course?",
            # "What are the two versions of the Math course offered in Terms 2 and 3?",
            # "What is the significance of hands-on projects in the Freshmore curriculum?",
            # "What is emphasized in the Global Humanities course?",
            # "What subjects are included in Term 2 of the Freshmore curriculum?",
            # "What foundational subjects are included in the Freshmore curriculum?",
            # "What is the focus of the Introduction to Programming course?",
            # "What are the two versions of the Math course offered in Term 2?",
            # "What is the focus of the Introduction to Human-Centred Design course?",
            # "What are the core subjects in Term 1 of the Freshmore curriculum?",
            # "What elective courses can students choose in Term 3?",
            # "What are the two versions of the Math course offered in Term 3?",
            # "What are the core subjects in Term 2 of the Freshmore curriculum?",
            # "What type of room are Freshmore students assigned to?",
            # "How are roommates matched in the hostel?",
            # "What security measures are in place at the hostel?",
            # "What amenities are available in the communal kitchen?",
            # "Where can students do laundry in the hostel?",
            # "What recreational facilities are available in the hostel?",
            # "Is there internet access in the hostel?",
            # "What is the purpose of live-in staff and student leaders?",
            # "What facilities are available on each floor of the hostel?",
            # "What is included in the communal kitchen on level 1?",
            # "What amenities are available in the pantry on level 9?",
            # "Where is the laundry room located?",
            # "What role do live-in staff and student leaders play?"
        ]
             

#--- ADD YOUR SOLUTION HERE (20 points)---
#---------------------------



In [47]:
import pandas as pd

results = []

for q in questions:
    retrieved_docs = retriever.invoke(q)
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])
    rag_prompt_text = rag_prompt.format(
        context=context,
        question=q
    )

    # get model to output d answer
    answer = openai_text(
        rag_prompt_text,
        model="gpt-4o-mini",
        max_output_tokens=300,
        temperature=0.2
    )

    # make into table
    results.append({
        "question": q,
        "answer": answer,
        "top_source_1": retrieved_docs[0].metadata.get("doc_id", "") if len(retrieved_docs) > 0 else "",
        "top_source_1_text": retrieved_docs[0].page_content if len(retrieved_docs) > 0 else "",
        "top_source_2": retrieved_docs[1].metadata.get("doc_id", "") if len(retrieved_docs) > 1 else "",
        "top_source_2_text": retrieved_docs[1].page_content if len(retrieved_docs) > 1 else "",
        "top_source_3": retrieved_docs[2].metadata.get("doc_id", "") if len(retrieved_docs) > 2 else "",
        "top_source_3_text": retrieved_docs[2].page_content if len(retrieved_docs) > 2 else "",
    })

results_df = pd.DataFrame(results)
results_df.to_csv("data/rag_test_results.csv", index=False)
results_df

,question,answer,top_source_1,top_source_1_text,top_source_2,top_source_2_text,top_source_3,top_source_3_text
0,What are the admissions deadlines for SUTD?,The application period for local Diploma appli...,admissions_undergraduate_admission-requirement...,…\nUndergraduate admissions\nAdmission require...,admissions_undergraduate_local-diploma_criteri...,"Notwithstanding the above, if you have a very ...",admissions_undergraduate_local-diploma_applica...,3. Outcome\nReceive an email notification by m...
1,Is there financial aid available?,"Yes, there is financial aid available at SUTD....",admissions_undergraduate_education-expenses_co...,Find out more:\nTuition fees and tuition grant...,admissions_undergraduate_education-expenses_fi...,…\nUndergraduate admissions\nEducation expense...,admissions_undergraduate_education-expenses_fe...,…\nEducation expenses\nTuition fees and tuitio...
2,What is the minimum score for the Mother Tongu...,The minimum score for the Mother Tongue Langua...,admissions_undergraduate_admission-requirement...,…\nUndergraduate admissions\nAdmission require...,admissions_undergraduate_admission-requirement...,Applicants who have been away from Singapore’s...,admissions_undergraduate_admission-requirement...,IELTS\nFor more information and registration f...
3,Do I require reference letters?,"Yes, you are required to provide reference let...",admissions_undergraduate_application-guide_con...,Recommendation/ Testimonial\nList up to two re...,admissions_undergraduate_application-guide_con...,Contact information\nProvide a valid email add...,admissions_undergraduate_admission-requirement...,a D7 grade for Higher MTL at Singapore-Cambrid...
4,Can polytechnic diploma students apply?,"Yes, polytechnic diploma students can apply to...",admissions_undergraduate_local-diploma_criteri...,…\nUndergraduate admissions\nLocal Diploma\nCr...,admissions_undergraduate_local-diploma_applica...,…\nUndergraduate admissions\nLocal Diploma\nAp...,admissions_undergraduate_local-diploma_criteri...,Final semester applicants\nSubmit results from...
5,Do I need SAT score?,SAT scores are optional for admission to SUTD....,admissions_undergraduate_admission-requirement...,Indonesian Ijazah SMA\nIndonesian Ujian Nasion...,admissions_undergraduate_local-diploma_criteri...,"Notwithstanding the above, if you have a very ...",admissions_undergraduate_admission-requirement...,IELTS\nFor more information and registration f...
6,How many PhD students does SUTD have?,The information regarding the number of PhD st...,about_design-ai_content,SUTD offers a truly unconventional education w...,admissions_undergraduate_local-diploma_applica...,Are all diplomas accepted for admission?\nGene...,admissions_undergraduate_local-diploma_criteri...,Final semester applicants\nSubmit results from...
7,How much are the tuition fees for Singaporeans?,The tuition fees for Singapore Citizens (SC) a...,admissions_undergraduate_education-expenses_fe...,Non-Subsidised Fee\n(Inclusive of GST)\nSingap...,admissions_undergraduate_education-expenses_fe...,Tuition Fees\nSubsidised Fee\nNon-Subsidised F...,admissions_undergraduate_education-expenses_fe...,(Inclusive of GST)\nSingapore Citizens\n(SC)\n...
8,How much are the tuition fees for internationa...,The tuition fees for international students at...,admissions_undergraduate_education-expenses_fe...,International Students\n(IS) (Inclusive of GST...,admissions_undergraduate_education-expenses_fe...,Tuition Fees\nSubsidised Fee\nNon-Subsidised F...,admissions_undergraduate_education-expenses_fe...,(Inclusive of GST)\nSingapore Citizens\n(SC)\n...
9,Is there a minimum CAP?,"Yes, there is a minimum Cumulative Average Poi...",istd_education_undergraduate_curriculum_content,Students are required to take a HASS elective ...,esd_education_undergraduate_curriculum_beyond-...,By the numbers\nESD Core (84 Credits)\nThe ESD...,admissions_undergraduate_admission-requirement...,Minimum requirements\nFind out more about the ...


### QUESTION: 


Manually inspect each answer, fact check whether the answer is correct (use Google or any other method) and check the retrieved documents

For each question, answer and context triple, record the following

- How accurate is the answer (1-5, 5 best)?
- How relevant is the retrieved context (1-5, 5 best)?
- How grounded is the answer in the retrieved context (instead of relying on the LLM's internal knowledge) (1-5, 5 best)?

**--- ADD YOUR SOLUTION HERE (20 points) ---**


------------------------------



You can try improve the chatbot by going back to previous steps in the notebook and change things until the submission deadline. For example, you can add more data sources, change the embedding models, change the data pre-processing, etc. 


# End

This concludes Individual assignment 2.

Please submit this notebook with your answers and the generated output cells as a **Jupyter notebook file** via github.

1. Create a private github repository **sutd_5055mlop** under your github user.
2. Add your instructors as collaborator: ddahlmeier, bearwithchris and MarkHershey
3. Save your submission as `individual_assignment_02_StudentID`.ipynb 
4. Push the submission files to your repo 
5. Submit the link to the repo via eDimensions


